In [1]:
# load data in registry
from lib.repository import DataRegistry

registry = DataRegistry()
registry.load_all()

Loading repo data...


Total progress:   0%|          | 0/14 [00:00<?, ?it/s]

Done in 67.92 seconds


In [ ]:
# import modules
from lib.builder import RaceContext, RaceDataRow
from lib.models import *


In [ ]:
def resolve_race_context(race: Race):
  competition = registry.competitions.get_by_id(race.CompetitionId)
  race_course = registry.race_courses.get_by_id(competition.RaceCourseId)
  horse_type = registry.horse_types.get_by_id(race.HorseTypeId)
  start_type = registry.race_start_types.get_by_id(race.RaceStartTypeId)
  race_context = RaceContext(competition, race, race_course, horse_type, start_type)
  return race_context

In [ ]:
skipped_races = 0

for item in registry.races.data_list[:10]:
  context = resolve_race_context(item)

  participants = registry.race_participants.get_participant_per_race(item.Id)
  if len(participants) <= 1:
    skipped_races += 1
    continue

  for p in participants:
    result = registry.race_results.get_result_by_participant_id(p.Id)
    raceCart = registry.race_cart_types.get_by_id(p.CartTypeId)

    driver = registry.drivers.get_by_id(p.DriverSourceId)
    driver_license = registry.driver_licenses.get_by_id(driver.DriverLicenseId)

    trainer = registry.drivers.get_by_id(p.TrainerSourceId)
    if trainer is None: trainer_license = None
    else: trainer_license = registry.driver_licenses.get_by_id(trainer.DriverLicenseId)

    horse = registry.horses.get_by_id(p.HorseSourceId)
    horseSex = registry.horse_sexes.get_by_id(horse.HorseSexId)
    horseType = registry.horse_types.get_by_id(horse.HorseTypeId)

    row_data = RaceDataRow(p, result)

    row_data.resolve_driver_data(driver, driver_license)
    row_data.resolve_trainer_data(trainer, trainer_license)
    row_data.resolve_horse_data(horse, horseSex.Sex, horseType.Type)
    row_data.resolve_cart(raceCart)

    context.race_rows.append(row_data)

  print(context)
